# register_101：寄存器里的数据

`T.load()` 之后数据就在 VGPR 里了。本实验回答三个问题：

1. 在 FlyDSL 里它们是什么类型（`fx.Vector`、`Float32` 这样的标量、寄存器 tensor）；
2. 这些类型自带哪些操作，各自变成什么指令；
3. 自带方法不够用时，怎样调用 `fx.rocdl`（DPP、ballot、fp8 转换……）或直接用 MLIR 的 `llvm` / `vector` dialect（任意 LLVM intrinsic、inline asm、插入元素）。

和 [LDS_101](lds_101.ipynb) 一样，每段 ISA 同时编译 gfx942（本仓库目标，MI300）和 gfx950（本机，MI350）并排对比；数值在本机 gfx950 上和 torch 对照。gfx942 的汇编只编译、没有在 gfx942 上跑过。

## 0. 准备

- `make_launcher(kernel_fn)`：把 `kernel_fn(A, O)` 包成 1 个 wave（64 lane）的 launcher；
- `row(T, tid, n)`：把 1D tensor 的第 `tid` 段 `n` 个元素取成一个子 tensor（每个 lane 一行）；
- `isa_both(build, ...)`：分别编译 gfx942 / gfx950，返回汇编全文；`side(...)` 并排打印；
- `alu(text)`：只留计算相关的指令，去掉 kernarg 读取、global 读写和 `s_waitcnt`，方便看寄存器上的操作；
- `ops(lines)`：按助记符计数。

In [1]:
import os
os.environ["FLYDSL_RUNTIME_ENABLE_CACHE"] = "0"   # 不用磁盘缓存，保证每次都重新 trace

import contextlib, glob, io, itertools, re, tempfile
from collections import Counter
from importlib.metadata import version

import torch
import flydsl.compiler as flyc
import flydsl.expr as fx
from flydsl._mlir import ir
from flydsl._mlir.dialects import llvm, vector

ARCHS = ("gfx942", "gfx950")
LOCAL_ARCH = torch.cuda.get_device_properties(0).gcnArchName.split(":")[0]
stream = fx.Stream(torch.cuda.current_stream().cuda_stream)
F32, BF16, I32 = fx.Float32, fx.BFloat16, fx.Int32
print("flydsl", version("flydsl"), "| torch", torch.__version__, "| 本机 GPU", LOCAL_ARCH)


def make_launcher(kernel_fn):
    kernel = flyc.kernel(known_block_size=(64, 1, 1))(kernel_fn)

    @flyc.jit
    def launch(A: fx.Tensor, O: fx.Tensor, stream: fx.Stream):
        kernel(A, O).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)

    return launch


def run(kernel_fn, A, O):
    with contextlib.redirect_stdout(io.StringIO()):   # trace 时的 print 只在 isa() 里显示一次
        make_launcher(kernel_fn)(A, O, stream)
    torch.cuda.synchronize()


def row(T, tid, n=8):
    return fx.make_view(fx.add_offset(fx.get_iter(T), tid * n), fx.make_layout(n, 1))


def isa(launcher, *args, arch="gfx942", quiet=False):
    dump_dir = tempfile.mkdtemp(prefix="reg101_")
    env = dict(ARCH=arch, COMPILE_ONLY="1", FLYDSL_DUMP_IR="1", FLYDSL_DUMP_DIR=dump_dir)
    old = {k: os.environ.get(k) for k in env}
    os.environ.update(env)
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            flyc.compile(launcher, *args)
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k)
            else:
                os.environ[k] = v
    for line in out.getvalue().splitlines():
        if not quiet and not line.startswith("[flydsl.compile]"):
            print(line)
    path, = glob.glob(f"{dump_dir}/**/*final_isa.s", recursive=True)
    return open(path).read()


def isa_both(kernel_fn, A, O):
    """分别编译 gfx942 / gfx950；trace 时的 print 只显示一次"""
    return {arch: isa(make_launcher(kernel_fn), A, O, stream, arch=arch, quiet=i > 0) for i, arch in enumerate(ARCHS)}


def body(text):
    lines, inside = [], False
    for raw in text.splitlines():
        if re.match(r"^\w+:", raw):
            inside = True
            continue
        line = raw.split(";")[0].strip()
        if inside and line and (not line.startswith(".") or line.startswith(".LBB")):
            lines.append(line)
        if line.startswith("s_endpgm"):
            break
    return lines


def alu(text):
    return [l for l in body(text) if not re.match(r"(s_load|global_|s_waitcnt|s_endpgm)", l)]


def ops(lines):
    return dict(Counter(l.split()[0] for l in lines))


def meta(text, key):
    return int(re.search(rf"\.{key}:\s+(\d+)", text).group(1))


def side(columns, width=44):
    print(" | ".join(f"{name:{width}}" for name in columns).rstrip())
    for r in itertools.zip_longest(*columns.values(), fillvalue=""):
        print(" | ".join(f"{c:{width}}" for c in r).rstrip())


def show(kernel_fn, A, O, width=44):
    """编译两种架构，并排打印计算指令，返回 {arch: 指令列表}"""
    texts = isa_both(kernel_fn, A, O)
    lines = {arch: alu(t) for arch, t in texts.items()}
    side(lines, width)
    return lines

flydsl 0.3.4.1 | torch 2.9.1+rocm7.2.0.git7e1940d4 | 本机 GPU gfx950


## 1. 寄存器里有哪几种东西

一个 lane 读 8 个 bf16，看 trace 时各种值的类型：

In [2]:
def show_types(A, O):
    tid = fx.thread_idx.x
    v = row(A, tid).load()
    print("row(A, tid).load()       :", v)
    print("v[0]                     :", type(v[0]).__name__, v[0].ir_value().type)
    print("A[tid * 8]               :", type(A[tid * 8]).__name__)
    print("fx.Float32(0.5)          :", repr(F32(0.5)), "is_static =", F32(0.5).is_static())
    frag = fx.make_rmem_tensor(8, F32)
    print("fx.make_rmem_tensor(8, F32):", frag.type)
    frag.store(v.to(F32))
    frag[0] = frag[0] + frag[7]
    row(O, tid).store(frag.load())


A = torch.randn(64 * 8, device="cuda").to(torch.bfloat16)
O = torch.zeros(64 * 8, device="cuda")
texts = isa_both(show_types, A, O)
side({arch: alu(t) + [f"scratch = {meta(t, 'private_segment_fixed_size')} B"] for arch, t in texts.items()})

run(show_types, A, O)
expect = A.float().view(64, 8).clone()
expect[:, 0] += expect[:, 7]
print(f"本机 {LOCAL_ARCH} GPU 结果正确:", torch.equal(O.view(64, 8), expect))

row(A, tid).load()       : Vector(vector<8xbf16> o (8,), BFloat16)
v[0]                     : BFloat16 bf16
A[tid * 8]               : BFloat16
fx.Float32(0.5)          : Float32(0.5) is_static = True
fx.make_rmem_tensor(8, F32): !fly.memref<f32, register, 8:1>
gfx942                                       | gfx950
v_lshlrev_b32_e32 v1, 4, v0                  | v_lshlrev_b32_e32 v1, 4, v0
v_lshlrev_b32_e32 v0, 5, v0                  | v_lshlrev_b32_e32 v0, 5, v0
v_lshlrev_b32_e32 v12, 16, v5                | v_lshlrev_b32_e32 v12, 16, v5
v_and_b32_e32 v11, 0xffff0000, v4            | v_and_b32_e32 v11, 0xffff0000, v4
v_lshlrev_b32_e32 v10, 16, v4                | v_lshlrev_b32_e32 v10, 16, v4
v_and_b32_e32 v13, 0xffff0000, v5            | v_and_b32_e32 v13, 0xffff0000, v5
v_lshlrev_b32_e32 v1, 16, v2                 | v_lshlrev_b32_e32 v1, 16, v2
v_and_b32_e32 v9, 0xffff0000, v3             | v_and_b32_e32 v9, 0xffff0000, v3
v_lshlrev_b32_e32 v8, 16, v3                 | v_lshlrev_b32_e

| 类型 | 从哪来 | IR | 说明 |
|---|---|---|---|
| `fx.Vector` | `T.load()`、`fx.full(...)`、`Vector.from_elements(...)` | `vector<8xbf16>` | 一个 MLIR 值，占若干 VGPR：8 个 bf16 = 4 个 VGPR（每个 VGPR 放 2 个） |
| 标量 `Numeric`（`Float32`、`BFloat16`、`Int32`…） | `v[i]`、`T[i]`、`fx.Float32(x)` | `f32` / `bf16` / `i32` | 运行时值占 1 个 VGPR（wave 内一致时编译器可能放进 SGPR） |
| 编译期常数 | Python `int`/`float`，或 `fx.Float32(0.5)`（`is_static() = True`） | 无 | 参与运算时折叠成立即数 |
| 寄存器 tensor | `fx.make_rmem_tensor(shape, dtype)`、`fx.make_fragment_like(t)` | `!fly.memref<f32, register, 8:1>` | 带 layout，能像 tensor 一样切片、`.load()/.store()`；编译后放进 VGPR，`scratch = 0` 说明没有落到内存 |

ISA 里看不到 "load 到寄存器" 这一步之外的任何搬运：`.to(F32)` 就是 `v_lshlrev_b32 16`（低半个 bf16）和 `v_and_b32 0xffff0000`（高半个 bf16），寄存器 tensor 的读写直接变成对这些 VGPR 的使用。

## 2. `fx.Vector` 自带的操作

`fx.Vector` 的方法（`flydsl/expr/typing.py`）：

| 类别 | 方法 |
|---|---|
| 逐元素运算 | `+ - * / // % **`、`& \| ^ ~ << >>`、`abs`、`-x`，标量自动 broadcast |
| 比较 / 选择 | `< <= > >= == !=` 得到 `Boolean` vector，`cond.select(a, b)` |
| 类型 | `.to(dtype)`（数值转换）、`.bitcast(dtype)`（按位重新解释） |
| 形状 | `v[i]`、`v[(None, j)]`（切片）、`.reshape(shape)`、`.broadcast_to(shape)`、`.shuffle(other, mask)` |
| 归约 | `.reduce("add" / "mul" / "max" / "min" / ...)` |
| 构造 | `fx.full(shape, value, dtype)`、`fx.zeros_like(v)`、`fx.Vector.from_elements([...])` |
| 数学函数 | `fx.math.exp2 / exp / log / sqrt / rsqrt / fma / tanh / ...`，对 vector 和标量都能用 |

### 2.1 f32 逐元素运算：packed 指令

`x * x + 1.0`，x 是 8 个 f32：

In [3]:
def f32_madd(A, O):
    tid = fx.thread_idx.x
    x = row(A, tid).load()
    row(O, tid).store(x * x + 1.0)


A = torch.randn(64 * 8, device="cuda")
O = torch.zeros_like(A)
show(f32_madd, A, O)
run(f32_madd, A, O)
print("GPU 正确:", torch.equal(O, A * A + 1))

gfx942                                       | gfx950
v_lshlrev_b32_e32 v12, 5, v0                 | v_lshlrev_b32_e32 v12, 5, v0
v_pk_mul_f32 v[8:9], v[2:3], v[2:3]          | v_pk_mul_f32 v[8:9], v[2:3], v[2:3]
v_pk_mul_f32 v[10:11], v[0:1], v[0:1]        | v_pk_mul_f32 v[10:11], v[0:1], v[0:1]
v_pk_mul_f32 v[2:3], v[6:7], v[6:7]          | v_pk_mul_f32 v[2:3], v[6:7], v[6:7]
v_pk_mul_f32 v[0:1], v[4:5], v[4:5]          | v_pk_mul_f32 v[0:1], v[4:5], v[4:5]
v_pk_add_f32 v[6:7], v[8:9], 1.0 op_sel_hi:[1,0] | v_pk_add_f32 v[6:7], v[8:9], 1.0 op_sel_hi:[1,0]
v_pk_add_f32 v[4:5], v[10:11], 1.0 op_sel_hi:[1,0] | v_pk_add_f32 v[4:5], v[10:11], 1.0 op_sel_hi:[1,0]
v_pk_add_f32 v[2:3], v[2:3], 1.0 op_sel_hi:[1,0] | v_pk_add_f32 v[2:3], v[2:3], 1.0 op_sel_hi:[1,0]
v_pk_add_f32 v[0:1], v[0:1], 1.0 op_sel_hi:[1,0] | v_pk_add_f32 v[0:1], v[0:1], 1.0 op_sel_hi:[1,0]
GPU 正确: True


- gfx942 / gfx950 都有 2×f32 的 packed 指令：8 个元素用 4 条 `v_pk_mul_f32` + 4 条 `v_pk_add_f32`，每条处理一对相邻 VGPR。常数 `1.0` 是内联立即数，`op_sel_hi:[1,0]` 表示两个 lane 用同一个常数。
- 编译器没有把乘加合成 FMA（没有 fastmath 时 `a*b+c` 和 `fma(a,b,c)` 舍入不同）；需要 FMA 时写 `fx.math.fma(x, x, 1.0)`。

### 2.2 类型转换：bf16 ↔ f32

三个 kernel：bf16 → f32、f32 → bf16、直接对 bf16 vector 做乘法。

In [4]:
def bf16_to_f32(A, O):
    tid = fx.thread_idx.x
    row(O, tid).store(row(A, tid).load().to(F32))


def f32_to_bf16(A, O):
    tid = fx.thread_idx.x
    row(O, tid).store(row(A, tid).load().to(BF16))


def bf16_mul(A, O):
    tid = fx.thread_idx.x
    x = row(A, tid).load()
    row(O, tid).store(x * x)


A16 = torch.randn(64 * 8, device="cuda").to(torch.bfloat16)
A32 = torch.randn(64 * 8, device="cuda")
cases = [("bf16 -> f32", bf16_to_f32, A16, torch.zeros(64 * 8, device="cuda"), lambda a: a.float()),
         ("f32 -> bf16", f32_to_bf16, A32, torch.zeros_like(A16), lambda a: a.to(torch.bfloat16)),
         ("bf16 * bf16", bf16_mul, A16, torch.zeros_like(A16), lambda a: (a.float() * a.float()).to(torch.bfloat16))]
for name, fn, A, O, ref in cases:
    run(fn, A, O)
    counts = {arch: ops(alu(t)) for arch, t in isa_both(fn, A, O).items()}
    print(f"{name}: GPU 正确 {torch.equal(O, ref(A))}")
    for arch, c in counts.items():
        print(f"    {arch}: {c}")

print("\nf32 -> bf16 的指令：")
show(f32_to_bf16, A32, torch.zeros_like(A16));

bf16 -> f32: GPU 正确 True
    gfx942: {'v_lshlrev_b32_e32': 6, 'v_and_b32_e32': 4}
    gfx950: {'v_lshlrev_b32_e32': 6, 'v_and_b32_e32': 4}
f32 -> bf16: GPU 正确 True
    gfx942: {'v_lshlrev_b32_e32': 2, 's_movk_i32': 1, 's_mov_b32': 1, 'v_bfe_u32': 8, 'v_or_b32_e32': 8, 'v_add3_u32': 8, 'v_cmp_u_f32_e32': 8, 'v_cndmask_b32_e32': 8, 'v_perm_b32': 4}
    gfx950: {'v_lshlrev_b32_e32': 2, 'v_cvt_pk_bf16_f32': 4}
bf16 * bf16: GPU 正确 True
    gfx942: {'v_lshlrev_b32_e32': 5, 's_movk_i32': 1, 's_mov_b32': 1, 'v_and_b32_e32': 4, 'v_mul_f32_e32': 8, 'v_bfe_u32': 8, 'v_or_b32_e32': 8, 'v_add3_u32': 8, 'v_cmp_u_f32_e32': 8, 'v_cndmask_b32_e32': 8, 'v_perm_b32': 4}
    gfx950: {'v_lshlrev_b32_e32': 5, 'v_and_b32_e32': 4, 'v_mul_f32_e32': 8, 'v_cvt_pk_bf16_f32': 4}

f32 -> bf16 的指令：
gfx942                                       | gfx950
v_lshlrev_b32_e32 v1, 5, v0                  | v_lshlrev_b32_e32 v1, 5, v0
s_movk_i32 s0, 0x7fff                        | v_lshlrev_b32_e32 v0, 4, v0
v_lshlrev_b32_e32

- bf16 → f32 不需要计算：bf16 就是 f32 的高 16 位，`v_lshlrev_b32 16` / `v_and_b32 0xffff0000` 把它放回高位即可。
- f32 → bf16 要做 round-to-nearest-even。gfx942 没有对应指令，每个元素用 `v_bfe_u32`（取第 16 位）+ `v_add3_u32`（加 0x7fff + 舍入位）+ `v_cmp_u_f32` / `v_or_b32` / `v_cndmask`（NaN 保持 quiet），最后 `v_perm_b32` 把两个高半字拼成一个 VGPR，8 个元素共 48 条指令。gfx950 新增 `v_cvt_pk_bf16_f32`，一条转两个，8 个元素 4 条。
- 对 bf16 vector 直接做 `x * x`：两种架构都没有 bf16 乘法，编译器自动转成 f32 计算再转回 bf16，所以代价主要在转回 bf16 上。要在 bf16 上做一串运算时，先 `.to(F32)` 算完再一次 `.to(BF16)`，结果和舍入次数都更可控。

### 2.2.1 gfx942 上 f32 → bf16 的简化写法

bf16 就是 f32 的高 16 位，所以舍入可以直接在 f32 的位模式上用整数运算完成：先给低 16 位加一个数决定是否进位，再取高 16 位。编译器默认的 `.to(BF16)` 还要保证 NaN 转换后仍是 NaN，如果输入里不会有 NaN、或者可以接受别的舍入方式，就可以省掉这部分：

| mode | 舍入方式 | 对位模式 `b` 的操作 |
|---|---|---|
| `rne` | 就近舍入到偶数（RNE），和 `.to(BF16)` 相同，但不处理 NaN | `b + 0x7FFF + ((b >> 16) & 1)` |
| `half_away` | 四舍五入，平局远离 0 | `b + 0x8000` |
| `up` | 向 +inf 舍入：正数只要低 16 位不为 0 就进位，负数直接截断 | `b + (((b >> 31) - 1) & 0xFFFF)` |
| `trunc` | 向 0 舍入（截断） | 不加，直接丢掉低 16 位 |

`up` 里 `b >> 31` 是符号位：正数时 `0 - 1 = 0xFFFFFFFF`，`& 0xFFFF` 得到 `0xFFFF`；负数时 `1 - 1 = 0`。

In [5]:
def f32_to_bf16_bits(x, mode):
    """在 f32 的位模式上做整数运算，取高 16 位作为 bf16"""
    b = x.bitcast(fx.Uint32)
    if mode == "rne":
        b = b + 0x7FFF + ((b >> 16) & 1)
    elif mode == "half_away":
        b = b + 0x8000
    elif mode == "up":
        b = b + (((b >> 31) - 1) & 0xFFFF)
    return (b >> 16).to(fx.Uint16).bitcast(BF16)       # mode == "trunc"：什么都不加


def make_cvt(mode):
    def kernel_fn(A: fx.Tensor, O: fx.Tensor):
        tid = fx.thread_idx.x
        x = row(A, tid).load()
        row(O, tid).store(x.to(BF16) if mode == "default" else f32_to_bf16_bits(x, mode))
    return kernel_fn


def ref_bits(A, mode):
    """torch 上用同样的整数运算算出参考结果"""
    b = A.view(torch.int32).long() & 0xFFFFFFFF
    add = {"rne": 0x7FFF + ((b >> 16) & 1), "half_away": 0x8000, "trunc": 0,
           "up": torch.where(b >> 31 == 0, 0xFFFF, 0)}[mode]
    return (((b + add) >> 16) & 0xFFFF).to(torch.int32).to(torch.int16).view(torch.bfloat16)


MODES = ["default", "rne", "half_away", "up", "trunc"]
A32 = torch.randn(64 * 8, device="cuda") * 100
O16 = torch.zeros(64 * 8, device="cuda", dtype=torch.bfloat16)
texts = {}
for mode in MODES:
    texts[mode] = isa_both(make_cvt(mode), A32, O16)
    run(make_cvt(mode), A32, O16)
    ref = A32.to(torch.bfloat16) if mode == "default" else ref_bits(A32, mode)
    counts = {arch: len(alu(t)) for arch, t in texts[mode].items()}
    print(f"{mode:10} 指令数 {counts} | GPU 等于参考: {torch.equal(O16, ref)!s:5} "
          f"| 等于 torch .to(bfloat16): {torch.equal(O16, A32.to(torch.bfloat16))}")

print("\ngfx942 的指令：")
side({mode: alu(texts[mode]["gfx942"]) for mode in ["rne", "up", "trunc"]}, width=36)

default    指令数 {'gfx942': 48, 'gfx950': 6} | GPU 等于参考: True  | 等于 torch .to(bfloat16): True
rne        指令数 {'gfx942': 24, 'gfx950': 24} | GPU 等于参考: True  | 等于 torch .to(bfloat16): True
half_away  指令数 {'gfx942': 15, 'gfx950': 15} | GPU 等于参考: True  | 等于 torch .to(bfloat16): True
up         指令数 {'gfx942': 31, 'gfx950': 31} | GPU 等于参考: True  | 等于 torch .to(bfloat16): False
trunc      指令数 {'gfx942': 7, 'gfx950': 7} | GPU 等于参考: True  | 等于 torch .to(bfloat16): False

gfx942 的指令：
rne                                  | up                                   | trunc
v_lshlrev_b32_e32 v1, 5, v0          | v_lshlrev_b32_e32 v1, 5, v0          | v_lshlrev_b32_e32 v1, 5, v0
s_movk_i32 s0, 0x7fff                | v_lshlrev_b32_e32 v10, 4, v0         | s_mov_b32 s0, 0x7060302
v_lshlrev_b32_e32 v10, 4, v0         | s_mov_b32 s0, 0x7060302              | v_lshlrev_b32_e32 v0, 4, v0
s_mov_b32 s1, 0x7060302              | v_lshrrev_b32_e32 v0, 31, v5         | v_perm_b32 v5, v5, v4, s0
v_bfe_u32 v0, v5, 16,

特殊值上的区别。表里是转换后 bf16 的位模式（十六进制）：

In [6]:
special = {
    "1+2^-8 (平局)":   0x3F808000,
    "-(1+2^-8)":      0xBF808000,
    "1+2^-9":         0x3F804000,
    "-(1+2^-9)":      0xBF804000,
    "最大 f32":        0x7F7FFFFF,
    "+inf":           0x7F800000,
    "quiet NaN":      0x7FC00000,
    "NaN 0x7f800001": 0x7F800001,
}
S = torch.zeros(64 * 8, dtype=torch.int32)
S[:len(special)] = torch.tensor([v - (1 << 32) if v >= 1 << 31 else v for v in special.values()], dtype=torch.int32)
S = S.view(torch.float32).cuda()
print(f"{'':16}" + "".join(f"{m:>11}" for m in MODES))
results = {}
for mode in MODES:
    run(make_cvt(mode), S, O16)
    results[mode] = O16.view(torch.int16)[:len(special)].tolist()
for i, name in enumerate(special):
    print(f"{name:16}" + "".join(f"{results[m][i] & 0xFFFF:>11x}" for m in MODES))

                    default        rne  half_away         up      trunc
1+2^-8 (平局)            3f80       3f80       3f81       3f81       3f80
-(1+2^-8)              bf80       bf80       bf81       bf80       bf80
1+2^-9                 3f80       3f80       3f80       3f81       3f80
-(1+2^-9)              bf80       bf80       bf80       bf80       bf80
最大 f32                 7f80       7f80       7f80       7f80       7f7f
+inf                   7f80       7f80       7f80       7f80       7f80
quiet NaN              7fc0       7fc0       7fc0       7fc0       7fc0
NaN 0x7f800001         7fc0       7f80       7f80       7f81       7f80


- 指令数（gfx942，含 3~4 条地址/常数准备）：默认 48 → `up` 31 → `rne` 24 → `half_away` 15 → `trunc` 7。
  - `rne`：每个元素只剩 `v_bfe_u32`（取第 16 位）+ `v_add3_u32`（三个数一次相加），省掉了 NaN 判断的 `v_cmp_u_f32` / `v_or_b32` / `v_cndmask`。
  - `up`：`(b >> 31) - 1` 是 `v_lshrrev` + `v_add -1`；`& 0xFFFF` 被编译器折叠进了 SDWA 加法（`v_add_u32_sdwa ... src0_sel:WORD_0`，只取低 16 位参与加法），每个元素 3 条。
  - `trunc`：不需要任何计算，两个元素只要一条 `v_perm_b32`（选择子 `0x07060302` 取两个寄存器的高半字）。
  - 所有写法最后都是 `v_perm_b32` 拼包：编译器从 `(b >> 16).to(Uint16)` 自动认出了这个模式。
- 随机数据上，`rne` 和 `half_away` 都和 torch 完全一致：两者只在恰好平局时不同（下表第一行 `1+2^-8`：RNE 取偶数 `3f80`，`half_away` 进位到 `3f81`），随机数据几乎碰不到平局。`up` 和 `trunc` 是另一种舍入方式，和 torch 不同是预期的，只和同样的整数参考对比。
- 特殊值：
  - 最大 f32 在 `rne` / `half_away` / `up` 下都变成 `+inf`（`7f80`），这是这些舍入方式下正确的溢出行为；`trunc` 保持最大有限值 `7f7f`。
  - quiet NaN（`7fc00000`，torch 产生的 NaN 都是这种）在所有写法下都还是 NaN。
  - 只有低 16 位非零的 NaN（`7f800001`）：截掉低位后变成 `7f80` 即 `+inf`，`rne` / `half_away` / `trunc` 都会出错；`up` 碰巧进位成 `7f81`，还是 NaN。默认写法把它变成 quiet NaN `7fc0`，这就是默认写法多出那些 `v_cmp_u_f32` / `v_or_b32` / `v_cndmask` 的原因。
- gfx950 上这 4 种写法的指令数和 gfx942 一样：它们只用了整数指令，编译器不会再换成 `v_cvt_pk_bf16_f32`。而 gfx950 上默认的 `.to(BF16)` 只要 6 条，所以这些技巧主要给 gfx942 用；截断在两种架构上都是最便宜的。

### 2.3 不产生计算的操作：bitcast / shuffle / 切片

`.bitcast` 只是重新解释寄存器，`.shuffle` / 切片只是选寄存器。下面交换每对相邻 bf16，并取出偶数位置的元素：

In [7]:
def free_ops(A, O):
    tid = fx.thread_idx.x
    x = row(A, tid).load()                                # 8 x bf16，4 个 VGPR
    print("x.bitcast(I32)          :", x.bitcast(I32))
    print("x.reshape((4, 2))[(None, 0)]:", x.reshape((4, 2))[(None, 0)])
    swapped = x.shuffle(x, [1, 0, 3, 2, 5, 4, 7, 6])
    row(O, tid).store(swapped)


A = torch.randn(64 * 8, device="cuda").to(torch.bfloat16)
O = torch.zeros_like(A)
show(free_ops, A, O)
run(free_ops, A, O)
print("GPU 正确:", torch.equal(O.view(-1, 2), A.view(-1, 2).flip(1)))

x.bitcast(I32)          : Vector(vector<4xi32> o (4,), Int32)
x.reshape((4, 2))[(None, 0)]: Vector(vector<4xbf16> o (4,), BFloat16)
gfx942                                       | gfx950
v_lshlrev_b32_e32 v4, 4, v0                  | v_lshlrev_b32_e32 v4, 4, v0
v_alignbit_b32 v3, v3, v3, 16                | v_alignbit_b32 v3, v3, v3, 16
v_alignbit_b32 v2, v2, v2, 16                | v_alignbit_b32 v2, v2, v2, 16
v_alignbit_b32 v1, v1, v1, 16                | v_alignbit_b32 v1, v1, v1, 16
v_alignbit_b32 v0, v0, v0, 16                | v_alignbit_b32 v0, v0, v0, 16
GPU 正确: True


- `bitcast` 和 `reshape` / 切片不生成指令。
- 交换同一个 VGPR 里的两个 bf16 是 `v_alignbit_b32 v, v, v, 16`（循环移位 16 位），每个 VGPR 一条；如果 shuffle 只是重新挑选整个 VGPR（例如交换两个 dword），就不需要任何指令。
- v0/v2 里 `a.bitcast(fx.Int16)` 再 `shuffle` 取 4 个元素喂 MFMA，就是用这些不生成指令的操作把寄存器整理成指令要的形状。

### 2.4 比较与选择

leaky ReLU：`(x > 0).select(x, x * 0.125)`

In [8]:
def leaky_relu(A, O):
    tid = fx.thread_idx.x
    x = row(A, tid).load()
    row(O, tid).store((x > 0.0).select(x, x * 0.125))


A = torch.randn(64 * 8, device="cuda")
O = torch.zeros_like(A)
lines = show(leaky_relu, A, O)
run(leaky_relu, A, O)
print("GPU 正确:", torch.equal(O, torch.where(A > 0, A, A * 0.125)))

gfx942                                       | gfx950
v_lshlrev_b32_e32 v16, 5, v0                 | v_lshlrev_b32_e32 v16, 5, v0
s_mov_b32 s0, 0x3e000000                     | s_mov_b32 s0, 0x3e000000
v_pk_mul_f32 v[10:11], v[0:1], s[0:1] op_sel_hi:[1,0] | v_pk_mul_f32 v[10:11], v[0:1], s[0:1] op_sel_hi:[1,0]
v_pk_mul_f32 v[14:15], v[4:5], s[0:1] op_sel_hi:[1,0] | v_pk_mul_f32 v[14:15], v[4:5], s[0:1] op_sel_hi:[1,0]
v_cmp_lt_f32_e32 vcc, 0, v5                  | v_cmp_lt_f32_e32 vcc, 0, v5
v_pk_mul_f32 v[12:13], v[6:7], s[0:1] op_sel_hi:[1,0] | v_pk_mul_f32 v[12:13], v[6:7], s[0:1] op_sel_hi:[1,0]
v_pk_mul_f32 v[8:9], v[2:3], s[0:1] op_sel_hi:[1,0] | v_pk_mul_f32 v[8:9], v[2:3], s[0:1] op_sel_hi:[1,0]
v_cndmask_b32_e32 v5, v15, v5, vcc           | v_cndmask_b32_e32 v5, v15, v5, vcc
v_cmp_lt_f32_e32 vcc, 0, v4                  | v_cmp_lt_f32_e32 vcc, 0, v4
s_nop 1                                      | s_nop 1
v_cndmask_b32_e32 v4, v14, v4, vcc           | v_cndmask_b32_e32 v4, v14, v

- 比较结果是每个 lane 1 位的 mask，存在 `vcc`（SGPR 对）里；`select` 是 `v_cndmask_b32`。每个元素一条 `v_cmp` + 一条 `v_cndmask`，中间还要 `s_nop 1` 等 `vcc` 写完。
- `x * 0.125` 仍然是 `v_pk_mul_f32`，常数放进 SGPR。

### 2.5 归约与 fastmath

`x.reduce("add")` 把一个 lane 内的 8 个元素加起来。浮点加法不满足结合律，默认只能按顺序加；`fastmath="reassoc"` 允许重排：

In [9]:
def reduce_add(A, O):
    tid = fx.thread_idx.x
    O[tid] = row(A, tid).load().reduce("add")


def reduce_add_reassoc(A, O):
    tid = fx.thread_idx.x
    O[tid] = row(A, tid).load().reduce("add", fastmath="reassoc")


A = torch.randn(64 * 8, device="cuda")
O = torch.zeros(64, device="cuda")
for fn in [reduce_add, reduce_add_reassoc]:
    print(f"==== {fn.__name__}")
    show(fn, A, O)
    run(fn, A, O)
    print("GPU 结果与 torch 接近:", torch.allclose(O, A.view(64, 8).sum(1), atol=1e-5))

==== reduce_add
gfx942                                       | gfx950
v_lshlrev_b32_e32 v1, 5, v0                  | v_lshlrev_b32_e32 v1, 5, v0
v_lshlrev_b32_e32 v0, 2, v0                  | v_lshlrev_b32_e32 v0, 2, v0
v_add_f32_e32 v1, 0, v2                      | v_add_f32_e32 v1, 0, v2
v_add_f32_e32 v1, v1, v3                     | v_add_f32_e32 v1, v1, v3
v_add_f32_e32 v1, v1, v4                     | v_add_f32_e32 v1, v1, v4
v_add_f32_e32 v1, v1, v5                     | v_add_f32_e32 v1, v1, v5
v_add_f32_e32 v1, v1, v6                     | v_add_f32_e32 v1, v1, v6
v_add_f32_e32 v1, v1, v7                     | v_add_f32_e32 v1, v1, v7
v_add_f32_e32 v1, v1, v8                     | v_add_f32_e32 v1, v1, v8
v_add_f32_e32 v1, v1, v9                     | v_add_f32_e32 v1, v1, v9
GPU 结果与 torch 接近: True
==== reduce_add_reassoc
gfx942                                       | gfx950
v_lshlrev_b32_e32 v1, 5, v0                  | v_lshlrev_b32_e32 v1, 5, v0
v_lshlrev_b32_e32 v0, 2, v0  

- 默认：8 条串行的 `v_add_f32`，第一条是归约初始值 `0 + v2`，之后每条都依赖上一条。
- `reassoc`：先用 `v_pk_add_f32` 两两相加（树形），只剩 5 条，依赖链也短。fastmath 也可以用 `with fx.arith.fastmath("reassoc"):` 包住一段代码统一设置。

### 2.6 数学函数：`fx.math` 与 `fx.rocdl`

硬件的 `v_exp_f32` 算的是 $2^x$，不处理 denormal。比较几种写法的指令数：

In [10]:
def exp2_default(A, O):
    tid = fx.thread_idx.x
    row(O, tid).store(fx.math.exp2(row(A, tid).load()))


def exp2_afn(A, O):
    tid = fx.thread_idx.x
    row(O, tid).store(fx.math.exp2(row(A, tid).load(), fastmath="afn"))


def exp_default(A, O):
    tid = fx.thread_idx.x
    row(O, tid).store(fx.math.exp(row(A, tid).load()))


def exp_via_rocdl_exp2(A, O):
    tid = fx.thread_idx.x
    x = row(A, tid).load()
    for i in fx.range_constexpr(8):                     # rocdl.exp2 只接受标量
        O[tid * 8 + i] = F32(fx.rocdl.exp2(F32.ir_type, (x[i] * 1.4426950408889634).ir_value()))


A = torch.randn(64 * 8, device="cuda") * 4
O = torch.zeros_like(A)
for fn, ref in [(exp2_default, A.exp2()), (exp2_afn, A.exp2()), (exp_default, A.exp()), (exp_via_rocdl_exp2, A.exp())]:
    run(fn, A, O)
    counts = {arch: len(alu(t)) for arch, t in isa_both(fn, A, O).items()}
    err = ((O - ref).abs() / ref.abs()).max().item()
    print(f"{fn.__name__:20} 指令数 {counts} | 最大相对误差 {err:.1e}")

print("\nexp2_afn：")
show(exp2_afn, A, O);

exp2_default         指令数 {'gfx942': 52, 'gfx950': 52} | 最大相对误差 0.0e+00
exp2_afn             指令数 {'gfx942': 9, 'gfx950': 9} | 最大相对误差 0.0e+00
exp_default          指令数 {'gfx942': 116, 'gfx950': 116} | 最大相对误差 0.0e+00
exp_via_rocdl_exp2   指令数 {'gfx942': 14, 'gfx950': 14} | 最大相对误差 6.0e-07

exp2_afn：
gfx942                                       | gfx950
v_lshlrev_b32_e32 v8, 5, v0                  | v_lshlrev_b32_e32 v8, 5, v0
v_exp_f32_e32 v0, v0                         | v_exp_f32_e32 v0, v0
v_exp_f32_e32 v4, v4                         | v_exp_f32_e32 v4, v4
v_exp_f32_e32 v5, v5                         | v_exp_f32_e32 v5, v5
v_exp_f32_e32 v6, v6                         | v_exp_f32_e32 v6, v6
v_exp_f32_e32 v7, v7                         | v_exp_f32_e32 v7, v7
v_exp_f32_e32 v1, v1                         | v_exp_f32_e32 v1, v1
v_exp_f32_e32 v2, v2                         | v_exp_f32_e32 v2, v2
v_exp_f32_e32 v3, v3                         | v_exp_f32_e32 v3, v3


- `fx.math.exp2` 默认会处理 denormal 结果：先判断范围、加偏移，`v_exp_f32` 之后再 `v_ldexp_f32` 修正，8 个元素 52 条。
- `fastmath="afn"`（允许近似函数）只剩一条 `v_exp_f32`。
- `fx.math.exp` 默认还要做 $e^x = 2^{x \log_2 e}$ 的高精度拆分（`v_fma_f32` + `v_rndne_f32` + `v_ldexp_f32` 等），8 个元素 116 条。
- `fx.rocdl.exp2` 直接对应 `v_exp_f32`，不加任何修正；`exp(x) ≈ rocdl.exp2(x * log2(e))` 是 softmax 里常用的写法，这里相对误差 5e-7，指令最少。代价是只接受标量，vector 要逐元素调用。
- 两种架构这一节完全相同。

## 3. 跨 lane 操作

`fx.Vector` 的方法都只在一个 lane 内部。要拿别的 lane 的数据，常用：

| 接口 | ISA | 说明 |
|---|---|---|
| `fx.gpu.shuffle_xor(v, off, width)`（还有 `shuffle_up/down/idx`） | `ds_swizzle_b32` / `ds_bpermute_b32` | 走 LDS 的交叉开关但不占 LDS 空间，用 `lgkmcnt` 等待 |
| `fx.rocdl.update_dpp(ty, old, src, ctrl, row_mask, bank_mask, bound_ctrl)` | `v_mov_b32_dpp` | 纯 VALU，只能在 16 lane 的 row 内按固定模式搬（quad_perm、row_shl/shr、row_mirror、row_half_mirror、row_bcast…） |
| `fx.rocdl.ds_bpermute(ty, byte_index, v)` | `ds_bpermute_b32` | 任意 lane 读任意 lane |
| `fx.rocdl.readfirstlane(ty, v)` / `readlane(ty, v, lane)` | `v_readfirstlane_b32` / `v_readlane_b32` | 结果放进 SGPR，整个 wave 共用 |
| `fx.rocdl.ballot(i64, pred)` | `v_cmp` 结果直接当 64 位 mask | 每个 lane 一位 |
| `fx.rocdl.mbcnt_lo / mbcnt_hi` | `v_mbcnt_lo/hi_u32_b32` | 数 mask 里比自己 lane 号小的 1 的个数（常用来算 lane id 或压缩下标） |
| `fx.rocdl.permlane16_swap / permlane32_swap` | `v_permlane16_swap_b32` 等 | gfx950 新增，两个半区之间交换；gfx950 上 `shuffle_xor(16/32)` 会自动用它（见下文） |

### 3.1 16 个 lane 求和：shuffle_xor 与 DPP

同样是 4 步蝶形求和（和 lane `l ^ 1`、`l ^ 2`、`l ^ 4`、`l ^ 8` 交换）：

In [11]:
def row_sum_shuffle(A, O):
    tid = fx.thread_idx.x
    v = A[tid]
    for off in [1, 2, 4, 8]:
        v = v + fx.gpu.shuffle_xor(v, off, 64)
    O[tid] = v


def dpp(v, ctrl):
    # old = src、row/bank mask 全开：每个 lane 都取 ctrl 指定的那个 lane
    return F32(fx.rocdl.update_dpp(F32.ir_type, v.ir_value(), v.ir_value(), ctrl, 0xF, 0xF, False))


def row_sum_dpp(A, O):
    tid = fx.thread_idx.x
    v = A[tid]
    # quad_perm [1,0,3,2] = l^1；quad_perm [2,3,0,1] = l^2；
    # row_half_mirror：8 个 lane 内倒序（此时每 4 个已相同，等价于 l^4）；row_mirror：16 个 lane 内倒序（等价于 l^8）
    for ctrl in [0xB1, 0x4E, 0x141, 0x140]:
        v = v + dpp(v, ctrl)
    O[tid] = v


A = torch.randn(64, device="cuda")
O = torch.zeros_like(A)
expect = A.view(4, 16).sum(1).repeat_interleave(16)
for fn in [row_sum_shuffle, row_sum_dpp]:
    print(f"==== {fn.__name__}")
    show(fn, A, O, width=60)
    run(fn, A, O)
    print("GPU 结果与 torch 接近:", torch.allclose(O, expect, atol=1e-5))

==== row_sum_shuffle
gfx942                                                       | gfx950
v_lshlrev_b32_e32 v0, 2, v0                                  | v_lshlrev_b32_e32 v0, 2, v0
ds_swizzle_b32 v2, v1 offset:swizzle(SWAP,1)                 | ds_swizzle_b32 v2, v1 offset:swizzle(SWAP,1)
v_add_f32_e32 v1, v1, v2                                     | v_add_f32_e32 v1, v1, v2
ds_swizzle_b32 v2, v1 offset:swizzle(SWAP,2)                 | ds_swizzle_b32 v2, v1 offset:swizzle(SWAP,2)
v_add_f32_e32 v1, v1, v2                                     | v_add_f32_e32 v1, v1, v2
ds_swizzle_b32 v2, v1 offset:swizzle(SWAP,4)                 | ds_swizzle_b32 v2, v1 offset:swizzle(SWAP,4)
v_add_f32_e32 v1, v1, v2                                     | v_add_f32_e32 v1, v1, v2
ds_swizzle_b32 v2, v1 offset:swizzle(SWAP,8)                 | ds_swizzle_b32 v2, v1 offset:swizzle(SWAP,8)
v_add_f32_e32 v1, v1, v2                                     | v_add_f32_e32 v1, v1, v2
GPU 结果与 torch 接近: True
==== row_su

- `shuffle_xor` 的偏移是常数且小于 32 时，编译器用 `ds_swizzle_b32 ... swizzle(SWAP, n)`，不需要算地址；但它是 LDS 单元的指令，有几十拍延迟并要 `lgkmcnt` 等待（汇编里过滤掉了 `s_waitcnt`）。
- DPP 版本全是 VALU：`v_mov_b32_dpp` 加一条 `v_add_f32`，DPP 读的寄存器刚被写过时要插 `s_nop 1`。这里编译器没有把 DPP 合并进 `v_add_f32_dpp`。
- DPP 的 `ctrl` 是编译期整数：`quad_perm` 是 4 个 2 位下标拼成的 8 位数（`[1,0,3,2]` = `0xB1`），`row_mirror` = `0x140`，`row_half_mirror` = `0x141`，`row_shl n` = `0x100+n`，`row_shr n` = `0x110+n`。

跨 16 个 lane 以上的交换，`shuffle_xor` 会换成别的指令：

In [12]:
def far_shuffle(A, O):
    tid = fx.thread_idx.x
    v = A[tid]
    for off in [16, 32]:
        v = v + fx.gpu.shuffle_xor(v, off, 64)
    O[tid] = v


O.zero_()
show(far_shuffle, A, O, width=60)
run(far_shuffle, A, O)
print("GPU 结果与 torch 接近:", torch.allclose(O, A.view(4, 16).sum(0).repeat(4), atol=1e-5))

gfx942                                                       | gfx950
v_lshlrev_b32_e32 v0, 2, v0                                  | v_lshlrev_b32_e32 v0, 2, v0
ds_swizzle_b32 v2, v1 offset:swizzle(SWAP,16)                | v_mov_b32_e32 v2, v1
v_add_f32_e32 v1, v1, v2                                     | v_mov_b32_e32 v3, v1
v_xor_b32_e32 v2, 0x80, v0                                   | s_nop 1
ds_bpermute_b32 v2, v2, v1                                   | v_permlane16_swap_b32_e32 v2, v3
v_add_f32_e32 v1, v1, v2                                     | v_cmp_eq_u32_e32 vcc, v2, v1
                                                             | s_nop 1
                                                             | v_cndmask_b32_e32 v2, v2, v3, vcc
                                                             | v_add_f32_e32 v1, v1, v2
                                                             | v_mov_b32_e32 v2, v1
                                                             | v_mov_b32

- gfx942：`l ^ 16` 仍在 32 个 lane 内，还能用 `ds_swizzle_b32`；`l ^ 32` 跨了两个半 wave，只能先算地址（`v_xor_b32 ... 0x80`，lane 号 × 4 字节）再 `ds_bpermute_b32`。
- gfx950：编译器改用新指令 `v_permlane16_swap_b32` / `v_permlane32_swap_b32`，再用 `v_cmp` + `v_cndmask` 挑出需要的那一半。全是 VALU，不走 LDS 单元。同一句 `shuffle_xor`，两种架构生成的指令完全不同。

### 3.2 ballot：整个 wave 的条件

数一下 64 个 lane 里有几个 `A[tid] > 0`：

In [13]:
def count_positive(A, O):
    tid = fx.thread_idx.x
    mask = fx.Int64(fx.rocdl.ballot(ir.IntegerType.get_signless(64), (A[tid] > 0.0).ir_value()))
    O[tid] = fx.math.ctpop(mask).to(I32)


A = torch.randn(64, device="cuda")
O = torch.zeros(64, device="cuda", dtype=torch.int32)
show(count_positive, A, O)
run(count_positive, A, O)
print("GPU 正确:", bool((O == (A > 0).sum()).all()), "| 个数", O[0].item())

gfx942                                       | gfx950
v_lshlrev_b32_e32 v0, 2, v0                  | v_lshlrev_b32_e32 v0, 2, v0
v_cmp_lt_f32_e32 vcc, 0, v1                  | v_cmp_lt_f32_e32 vcc, 0, v1
s_bcnt1_i32_b64 s0, vcc                      | s_bcnt1_i32_b64 s0, vcc
v_mov_b32_e32 v1, s0                         | v_mov_b32_e32 v1, s0
GPU 正确: True | 个数 29


- `ballot` 不需要额外指令：`v_cmp` 写出的 `vcc` 本身就是 64 位 mask；`ctpop` 是标量指令 `s_bcnt1_i32_b64`，结果在 SGPR，整个 wave 只算一次。

## 4. 专用指令：以 fp8 打包为例

`fx.rocdl` 里有大量直接对应硬件指令的 op（`fx.rocdl.mfma_*`、`cvt_pk_*`、`perm_b32`、`dot4_f32_fp8_fp8`、`fmed3`、`sched_*`、`s_waitcnt` ……），参数大多是 MLIR 类型和 `ir.Value`，需要自己 `.ir_value()` 和包回 `Numeric`。

把 4 个 f32 转成 fp8 并打包进一个 32 位：`cvt_pk_fp8_f32` 一次转两个，写进结果的低半字（`word_sel=False`）或高半字（`True`）：

In [14]:
def pack_fp8(A, O):
    tid = fx.thread_idx.x
    x = row(A, tid, 4).load()
    i32 = ir.IntegerType.get_signless(32)
    w = fx.rocdl.cvt_pk_fp8_f32(i32, x[0].ir_value(), x[1].ir_value(), I32(0).ir_value(), False)
    w = fx.rocdl.cvt_pk_fp8_f32(i32, x[2].ir_value(), x[3].ir_value(), w, True)
    O[tid] = I32(w)


A = torch.randn(64 * 4, device="cuda") * 8
O = torch.zeros(64, device="cuda", dtype=torch.int32)
show(pack_fp8, A, O)
run(pack_fp8, A, O)
got = O.view(torch.uint8).view(64, 4)
for name, dt in [("float8_e4m3fn  (OCP)", torch.float8_e4m3fn), ("float8_e4m3fnuz (FNUZ)", torch.float8_e4m3fnuz)]:
    print(f"本机 {LOCAL_ARCH} 结果等于 torch {name}:", torch.equal(got, A.view(64, 4).to(dt).view(torch.uint8)))

gfx942                                       | gfx950
v_lshlrev_b32_e32 v1, 4, v0                  | v_lshlrev_b32_e32 v1, 4, v0
v_lshlrev_b32_e32 v0, 2, v0                  | v_lshlrev_b32_e32 v0, 2, v0
v_mov_b32_e32 v1, 0                          | v_mov_b32_e32 v1, 0
v_cvt_pk_fp8_f32 v1, v2, v3                  | v_cvt_pk_fp8_f32 v1, v2, v3
v_cvt_pk_fp8_f32 v1, v4, v5 op_sel:[0,0,1]   | v_cvt_pk_fp8_f32 v1, v4, v5 op_sel:[0,0,1]
本机 gfx950 结果等于 torch float8_e4m3fn  (OCP): True
本机 gfx950 结果等于 torch float8_e4m3fnuz (FNUZ): False


- 两种架构都是两条 `v_cvt_pk_fp8_f32`，第二条用 `op_sel:[0,0,1]` 写高半字；ISA 看起来完全一样。
- 但格式不同：gfx942 的 fp8 是 FNUZ（`torch.float8_e4m3fnuz`），gfx950 改成了 OCP 标准（`torch.float8_e4m3fn`）。本机 gfx950 的结果和 `float8_e4m3fn` 一致、和 FNUZ 不一致。同一段代码在两种架构上数值不同，参考结果要按架构选。

## 5. 没有封装时：直接用 MLIR dialect

FlyDSL 的 `Vector` / `Numeric` 底层就是 MLIR 的 `ir.Value`，所以可以随时用 `.ir_value()` 拿出来，交给 MLIR 其它 dialect 的 op，再用 `fx.Int32(...)`、`fx.Vector(...)` 包回来。常用的三种：

| 需求 | 写法 |
|---|---|
| 调用任意 LLVM intrinsic | `llvm.call_intrinsic(结果类型, "llvm.amdgcn.xxx", [参数...], [], [])` |
| 改 vector 的一个元素（`fx.Vector` 没有对应方法） | `vector.insert(值, vec, [], [位置])` |
| 编译器不认识的指令、或想在 ISA 里做记号 | `llvm.inline_asm(结果类型, [参数...], "汇编", "约束", has_side_effects=...)` |

### 5.1 `llvm.call_intrinsic` 与 `vector.insert`

`llvm.amdgcn.ubfe.i32` 是无符号位段提取（`(x >> 4) & 0xff`），常用来拆 int4/fp4；另一个 kernel 把 8 个 f32 的第 5 个改成 3.0：

In [15]:
def bitfield_extract(A, O):
    tid = fx.thread_idx.x
    i32 = ir.IntegerType.get_signless(32)
    r = llvm.call_intrinsic(i32, "llvm.amdgcn.ubfe.i32", [A[tid].ir_value(), I32(4).ir_value(), I32(8).ir_value()], [], [])
    O[tid] = I32(r)


def insert_element(A, O):
    tid = fx.thread_idx.x
    x = row(A, tid).load()
    row(O, tid).store(fx.Vector(vector.insert(F32(3.0).ir_value(), x, [], [5])))


Ai = torch.randint(0, 1 << 20, (64,), device="cuda", dtype=torch.int32)
Oi = torch.zeros_like(Ai)
show(bitfield_extract, Ai, Oi)
run(bitfield_extract, Ai, Oi)
print("GPU 正确:", torch.equal(Oi, (Ai >> 4) & 0xFF))

A = torch.randn(64 * 8, device="cuda")
O = torch.zeros_like(A)
show(insert_element, A, O)
run(insert_element, A, O)
expect = A.view(64, 8).clone()
expect[:, 5] = 3.0
print("GPU 正确:", torch.equal(O.view(64, 8), expect))

gfx942                                       | gfx950
v_lshlrev_b32_e32 v0, 2, v0                  | v_lshlrev_b32_e32 v0, 2, v0
v_bfe_u32 v1, v1, 4, 8                       | v_bfe_u32 v1, v1, 4, 8
GPU 正确: True
gfx942                                       | gfx950
v_lshlrev_b32_e32 v4, 5, v0                  | v_lshlrev_b32_e32 v4, 5, v0
v_mov_b32_e32 v1, 0x40400000                 | v_mov_b32_e32 v1, 0x40400000
GPU 正确: True


- `ubfe` 变成一条 `v_bfe_u32 v1, v1, 4, 8`。intrinsic 的名字和参数类型可以查 LLVM 的 `IntrinsicsAMDGPU.td`；FlyDSL 自己的 `fx.rocdl.cdna4.cvt_sr_f32_to_bf16` 也是这样实现的。
- `vector.insert` 之后只多了一条把常数 3.0 放进 VGPR 的 `v_mov_b32`，其余元素原样写出。

### 5.2 inline asm

两个用途：

1. 在 ISA 里做记号：一条带 side effect 的 `s_nop`，方便在大 kernel 的汇编里找到某段代码；
2. 编译器还没有 intrinsic 的指令：FlyDSL 的 `fx.rocdl.inline_asm.cvt_pk_bf16_f32` 就是用 inline asm 发出 gfx950 的 `v_cvt_pk_bf16_f32`。约束串 `"=v,v,v"` 表示结果和两个输入都是 VGPR。

In [16]:
def asm_demo(A, O):
    tid = fx.thread_idx.x
    x = row(A, tid, 2).load()
    llvm.inline_asm(None, [], "s_nop 7 ; >>> marker", "", has_side_effects=True)
    O[tid] = I32(fx.rocdl.inline_asm.cvt_pk_bf16_f32(x[0].ir_value(), x[1].ir_value()))


A = torch.randn(64 * 2, device="cuda")
O = torch.zeros(64, device="cuda", dtype=torch.int32)
for arch in ARCHS:
    try:
        lines = alu(isa(make_launcher(asm_demo), A, O, stream, arch=arch))
        print(f"{arch}:", lines)
    except Exception as e:
        # 汇编器的具体报错写在 stderr（instruction not supported on this GPU），异常里只有 lld 失败
        print(f"{arch}: 编译失败 ->", type(e).__name__, [l.strip() for l in str(e).splitlines() if "error:" in l][0])

run(asm_demo, A, O)
print("本机 GPU 正确:", torch.equal(O.view(torch.bfloat16), A.to(torch.bfloat16)))

gfx942: 编译失败 -> DSLCompileError error: "-":2:3: lld invocation failed
gfx950: ['v_lshlrev_b32_e32 v1, 3, v0', 'v_lshlrev_b32_e32 v0, 2, v0', 's_nop 7', 'v_cvt_pk_bf16_f32 v1, v2, v3']
本机 GPU 正确: True


- gfx950：`s_nop 7` 记号和 `v_cvt_pk_bf16_f32` 都原样出现在汇编里，结果和 torch 的 bf16 转换一致。
- gfx942：没有这条指令，编译器不会检查 inline asm 的内容，直到汇编器才报 `instruction not supported on this GPU (gfx942): v_cvt_pk_bf16_f32`（打在 stderr 上），Python 侧收到的是 `DSLCompileError: lld invocation failed`。
- inline asm 对编译器是黑盒：不参与指令选择和优化，调度器也不知道它的延迟。只在确实没有 intrinsic、或者需要固定某段指令时使用。

## 总结

| 问题 | 结论 |
|---|---|
| `.load()` 得到什么 | `fx.Vector`（`vector<N x dtype>`，按 32 位打包进 VGPR）；`v[i]` / `T[i]` 得到标量 `Numeric`；`make_rmem_tensor` / `make_fragment_like` 得到带 layout 的寄存器 tensor，编译后也在 VGPR 里 |
| 逐元素运算 | 运算符 + 标量 broadcast；f32 自动用 `v_pk_mul/add_f32`；不自动合成 FMA，需要时 `fx.math.fma` |
| 类型转换 | bf16→f32 只是移位；f32→bf16 在 gfx942 上默认每元素约 6 条（RNE + NaN 处理），gfx950 用 `v_cvt_pk_bf16_f32`；bf16 运算都是先转 f32 |
| gfx942 上简化 f32→bf16 | 在位模式上加一个数再取高 16 位：RNE `b + 0x7FFF + ((b>>16)&1)`、四舍五入 `b + 0x8000`、向 +inf `b + (((b>>31)-1)&0xFFFF)`、截断（一条 `v_perm_b32` 拼两个）；不处理 NaN 时 8 个元素从 48 条降到 24 / 15 / 31 / 7 条 |
| 不生成指令的操作 | `bitcast`、`reshape`、切片、按 VGPR 重排的 `shuffle`；同一 VGPR 内交换 16 位要 `v_alignbit` |
| 选择 / 归约 | `cond.select` → `v_cmp` + `v_cndmask`；`reduce` 默认串行，`fastmath="reassoc"` 变成树形 + packed 加法 |
| 数学函数 | `fx.math.exp2/exp` 默认带 denormal / 精度处理；`fastmath="afn"` 或 `fx.rocdl.exp2` 只剩 `v_exp_f32` |
| 跨 lane | `fx.gpu.shuffle_xor` → gfx942 用 `ds_swizzle` / `ds_bpermute`（LDS 单元），gfx950 跨半区时用 `v_permlane*_swap`；`fx.rocdl.update_dpp` → `v_mov_b32_dpp`（VALU，row 内固定模式）；`ballot` / `readlane` / `readfirstlane` → SGPR |
| 专用指令 | `fx.rocdl.*` 直接对应硬件指令，参数用 `ir.Value`；fp8 指令在 gfx942 是 FNUZ、gfx950 是 OCP，数值不同 |
| 没有封装时 | `.ir_value()` 取出 MLIR 值：`llvm.call_intrinsic` 调任意 `llvm.amdgcn.*`；`vector.insert` 改单个元素；`llvm.inline_asm` 发任意指令或做记号（编译器不检查、不优化） |